In [23]:
import duckdb
import pandas as pd

con = duckdb.connect()

file_path = r"C:\Users\Admin\Healthcare data analytics\Data\healthcare_cleaned.csv"

con.execute(f"""
    CREATE OR REPLACE TABLE healthcare AS
    SELECT *
    FROM read_csv_auto('{file_path}')
""")

print("Healthcare table created successfully!")

Healthcare table created successfully!


In [24]:
result = con.execute("""
    SELECT COUNT(*) AS Total_Records
    FROM healthcare
""").df()

display(result)

,Total_Records
0,54860


In [25]:
columns = con.execute("""
    DESCRIBE healthcare
""").df()

display(columns)

,column_name,column_type,null,key,default,extra
0,Patient_Name,VARCHAR,YES,None,None,None
1,Age,BIGINT,YES,None,None,None
2,Gender,VARCHAR,YES,None,None,None
3,Blood_Type,VARCHAR,YES,None,None,None
4,Medical_Condition,VARCHAR,YES,None,None,None
5,Admission_Date,DATE,YES,None,None,None
6,Doctor,VARCHAR,YES,None,None,None
7,Hospital,VARCHAR,YES,None,None,None
8,Insurance_Provider,VARCHAR,YES,None,None,None
9,Billing_Amount,DOUBLE,YES,None,None,None


In [26]:
# PATIENT COUNT KPI

In [27]:
patient_count = con.execute("""
    SELECT COUNT(*) AS Patient_Count
    FROM healthcare
""").df()

display(patient_count)

,Patient_Count
0,54860


In [28]:
# RECOVERY RATE KPI 

In [29]:
recovery_rate = con.execute("""
    SELECT
        ROUND(AVG(Recovery_Flag) * 100, 2) AS Recovery_Rate
    FROM healthcare
""").df()

display(recovery_rate)

,Recovery_Rate
0,33.36


In [30]:
# AVGERAGE TREATMENT COST

In [31]:
avg_cost = con.execute("""
    SELECT
        ROUND(AVG(Billing_Amount), 2) AS Average_Treatment_Cost
    FROM healthcare
""").df()

display(avg_cost)

,Average_Treatment_Cost
0,25594.63


In [32]:
# AVERAGE LENGTH OF STAY

In [33]:
avg_stay = con.execute("""
    SELECT
        ROUND(AVG(Length_of_Stay), 2) AS Average_Length_of_Stay
    FROM healthcare
""").df()

display(avg_stay)

,Average_Length_of_Stay
0,15.5


In [34]:
#MAIN KPI QUERY

In [35]:
kpi = con.execute("""
    SELECT
        COUNT(*) AS Patient_Count,

        ROUND(AVG(Recovery_Flag) * 100, 2)
            AS Recovery_Rate,

        ROUND(AVG(Billing_Amount), 2)
            AS Average_Treatment_Cost,

        ROUND(AVG(Length_of_Stay), 2)
            AS Average_Length_of_Stay,

        COUNT(DISTINCT Hospital)
            AS Hospital_Count,

        COUNT(DISTINCT Doctor)
            AS Doctor_Count

    FROM healthcare
""").df()

display(kpi)

,Patient_Count,Recovery_Rate,Average_Treatment_Cost,Average_Length_of_Stay,Hospital_Count,Doctor_Count
0,54860,33.36,25594.63,15.5,39815,40276


In [36]:
# PATIENT DEMOGRAPHICS GENDER

In [37]:
gender = con.execute("""
    SELECT
        Gender,
        COUNT(*) AS Patient_Count
    FROM healthcare
    GROUP BY Gender
    ORDER BY Patient_Count DESC
""").df()

display(gender)

,Gender,Patient_Count
0,Male,27449
1,Female,27411


In [38]:
# PATIENT DEMOGRAPHICS AGE GROUPS

In [39]:
age_groups = con.execute("""
    SELECT
        Age_Group,
        COUNT(*) AS Patient_Count
    FROM healthcare
    GROUP BY Age_Group
    ORDER BY Patient_Count DESC
""").df()

display(age_groups)

,Age_Group,Patient_Count
0,46-60,12248
1,31-45,12088
2,61-75,12079
3,19-30,9504
4,76+,8056
5,0-18,885


In [40]:
# DISEASE ANALYSIS

In [41]:
disease = con.execute("""
    SELECT
        Medical_Condition,
        COUNT(*) AS Patient_Count
    FROM healthcare
    GROUP BY Medical_Condition
    ORDER BY Patient_Count DESC
""").df()

display(disease)

,Medical_Condition,Patient_Count
0,Arthritis,9207
1,Diabetes,9197
2,Hypertension,9131
3,Obesity,9127
4,Cancer,9121
5,Asthma,9077


In [42]:
# RECOVERY RATE BY DISEASE

In [43]:
disease_recovery = con.execute("""
    SELECT
        Medical_Condition,
        COUNT(*) AS Patient_Count,
        ROUND(AVG(Recovery_Flag) * 100, 2) AS Recovery_Rate
    FROM healthcare
    GROUP BY Medical_Condition
    ORDER BY Recovery_Rate DESC
""").df()

display(disease_recovery)

,Medical_Condition,Patient_Count,Recovery_Rate
0,Asthma,9077,34.30
1,Hypertension,9131,33.96
2,Diabetes,9197,33.26
3,Obesity,9127,33.14
4,Cancer,9121,33.01
5,Arthritis,9207,32.51


In [44]:
# HOSPITAL PERFORMANCE 

In [45]:
hospital_performance = con.execute("""
    SELECT
        Hospital,
        COUNT(*) AS Patient_Count,
        ROUND(AVG(Billing_Amount), 2) AS Average_Treatment_Cost,
        ROUND(AVG(Length_of_Stay), 2) AS Average_Length_of_Stay,
        ROUND(AVG(Recovery_Flag) * 100, 2) AS Recovery_Rate
    FROM healthcare
    GROUP BY Hospital
    ORDER BY Patient_Count DESC
""").df()

display(hospital_performance)

,Hospital,Patient_Count,Average_Treatment_Cost,Average_Length_of_Stay,Recovery_Rate
0,LLC Smith,44,23413.41,15.57,31.82
1,Ltd Smith,39,25727.32,16.51,30.77
2,Smith Ltd,37,26217.19,15.14,48.65
3,Johnson PLC,37,29229.12,16.14,24.32
4,Smith PLC,36,28595.12,17.64,38.89
...,...,...,...,...,...
39810,Hodge Group,1,47202.57,17.00,0.00
39811,"Hernandez and Erickson May,",1,10493.57,29.00,0.00
39812,"Rodriguez Graves, and Pennington",1,26970.53,18.00,0.00
39813,"Cox Jenkins, and Morrison",1,33210.72,28.00,100.00


In [46]:
# DOCTOR PERFROMANCE

In [47]:
doctor_performance = con.execute("""
    SELECT
        Doctor,
        COUNT(*) AS Patient_Count,
        ROUND(AVG(Billing_Amount), 2) AS Average_Billing,
        ROUND(AVG(Length_of_Stay), 2) AS Average_Length_of_Stay,
        ROUND(AVG(Recovery_Flag) * 100, 2) AS Recovery_Rate
    FROM healthcare
    GROUP BY Doctor
    ORDER BY Patient_Count DESC
""").df()

display(doctor_performance.head(10))

,Doctor,Patient_Count,Average_Billing,Average_Length_of_Stay,Recovery_Rate
0,Michael Smith,27,29055.62,15.63,22.22
1,John Smith,22,27732.25,14.36,40.91
2,Robert Smith,21,29007.65,13.81,33.33
3,Michael Johnson,20,23040.95,18.70,65.00
4,Robert Johnson,19,27589.11,13.79,26.32
5,David Smith,19,24912.93,15.32,36.84
6,James Smith,19,24313.35,17.68,26.32
7,Michael Williams,18,17171.73,15.50,50.00
8,Christopher Smith,17,19929.48,13.12,23.53
9,Matthew Smith,17,24790.30,13.76,64.71


In [48]:
# ADMISSION PATTERNS

In [49]:
admission_analysis = con.execute("""
    SELECT
        Admission_Type,
        COUNT(*) AS Patient_Count,
        ROUND(
            COUNT(*) * 100.0 /
            (SELECT COUNT(*) FROM healthcare),
            2
        ) AS Percentage
    FROM healthcare
    GROUP BY Admission_Type
    ORDER BY Patient_Count DESC
""").df()

display(admission_analysis)

,Admission_Type,Patient_Count,Percentage
0,Elective,18437,33.61
1,Urgent,18353,33.45
2,Emergency,18070,32.94


In [50]:
# MONTHLY ADMISSION TREND

In [51]:
monthly_admissions = con.execute("""
    SELECT
        Admission_Year_Month,
        COUNT(*) AS Patient_Count
    FROM healthcare
    GROUP BY Admission_Year_Month
    ORDER BY Admission_Year_Month
""").df()

display(monthly_admissions)

,Admission_Year_Month,Patient_Count
0,2019-05,672
1,2019-06,897
2,2019-07,950
3,2019-08,982
4,2019-09,924
...,...,...
56,2024-01,903
57,2024-02,865
58,2024-03,900
59,2024-04,939


In [52]:
# HIGH COST CASE

In [53]:
high_cost = con.execute("""
    SELECT
        Patient_Name,
        Medical_Condition,
        Hospital,
        Billing_Amount,
        Length_of_Stay
    FROM healthcare
    ORDER BY Billing_Amount DESC
    LIMIT 10
""").df()

display(high_cost)

,Patient_Name,Medical_Condition,Hospital,Billing_Amount,Length_of_Stay
0,Todd Carrillo,Hypertension,Griffin Group,52764.276736,26
1,Karen Kline,Cancer,Hernandez-Morton,52373.032374,14
2,Karen Kline,Cancer,Hernandez-Morton,52373.032374,14
3,David Sandoval,Hypertension,Sons and Bailey,52271.663747,9
4,Kathryn Gonzales,Diabetes,Miller Ltd,52211.852966,1
5,Brett Marshall,Asthma,PLC Garner,52181.837792,7
6,Laurie Hood,Arthritis,Walker-Garcia,52170.036854,2
7,Laurie Hood,Arthritis,Walker-Garcia,52170.036854,2
8,Justin Clark,Cancer,Ruiz-Anthony,52154.237722,23
9,Scott Powell,Cancer,George-Gonzalez,52102.240889,9


In [54]:
# LONGEST HOSPITAL STAYS

In [55]:
long_stays = con.execute("""
    SELECT
        Patient_Name,
        Medical_Condition,
        Hospital,
        Length_of_Stay,
        Billing_Amount
    FROM healthcare
    ORDER BY Length_of_Stay DESC
    LIMIT 10
""").df()

display(long_stays)

,Patient_Name,Medical_Condition,Hospital,Length_of_Stay,Billing_Amount
0,Andrew Watts,Diabetes,"Hernandez Rogers and Vang,",30,37909.782410
1,Christopher Berg,Cancer,Padilla-Walker,30,19784.631062
2,Sean Jennings,Diabetes,Clark-Johnson,30,20257.544283
3,April Valencia,Diabetes,Levine-Miller,30,22356.226492
4,Rita Archer,Diabetes,"and Marquez Silva Smith,",30,48995.980592
5,Jennifer Mcmillan,Obesity,Rodriguez and Sons,30,15475.403237
6,Christina Martin,Diabetes,Inc Lee,30,17670.505217
7,Alan Taylor,Diabetes,"Hunt, Carlson and Cherry",30,36339.819081
8,Ashley Webb Dds,Cancer,"Gordon, Fox Lane and",30,40469.566259
9,Lauren Ramirez,Asthma,Harrison-Parker,30,1791.389001


In [56]:
# DATA QUALITY CHECK

In [57]:
quality_check = con.execute("""
    SELECT
        COUNT(*) AS Total_Records,

        SUM(CASE WHEN Age < 0 OR Age > 120 THEN 1 ELSE 0 END)
            AS Invalid_Age,

        SUM(CASE WHEN Billing_Amount < 0 THEN 1 ELSE 0 END)
            AS Invalid_Billing,

        SUM(CASE WHEN Length_of_Stay < 0 THEN 1 ELSE 0 END)
            AS Invalid_Length_of_Stay,

        SUM(CASE WHEN Recovery_Flag NOT IN (0, 1) THEN 1 ELSE 0 END)
            AS Invalid_Recovery_Flag

    FROM healthcare
""").df()

display(quality_check)

,Total_Records,Invalid_Age,Invalid_Billing,Invalid_Length_of_Stay,Invalid_Recovery_Flag
0,54860,0.0,0.0,0.0,0.0


In [59]:
# Readmission Rate
#A reliable readmission rate could not be calculated because the dataset does not contain a dedicated readmission indicator or sufficient longitudinal patient admission information.

In [58]:
final_kpis = con.execute("""
    SELECT
        COUNT(*) AS Patient_Count,

        ROUND(AVG(Recovery_Flag) * 100, 2)
            AS Recovery_Rate,

        ROUND(AVG(Billing_Amount), 2)
            AS Average_Treatment_Cost,

        ROUND(AVG(Length_of_Stay), 2)
            AS Average_Length_of_Stay,

        COUNT(DISTINCT Hospital)
            AS Hospital_Count,

        COUNT(DISTINCT Doctor)
            AS Doctor_Count,

        COUNT(DISTINCT Medical_Condition)
            AS Disease_Count

    FROM healthcare
""").df()

display(final_kpis)

,Patient_Count,Recovery_Rate,Average_Treatment_Cost,Average_Length_of_Stay,Hospital_Count,Doctor_Count,Disease_Count
0,54860,33.36,25594.63,15.5,39815,40276,6
